# 02 - Limpieza, Transformación y Feature Engineering

## Objetivo

Construir una versión limpia y enriquecida del dataset de siniestros viales, preservando el dato crudo y generando variables aptas para análisis y modelado.

## Actividades realizadas

- Carga del archivo fuente desde `data/raw/`.
- Normalización de nombres, valores faltantes y tipos de datos.
- Creación de variables derivadas temporales, etarias y de severidad.
- Validación estructural del resultado y persistencia del dataset procesado.

## Entradas

- `data/raw/siniestros_viales_víctimas.xlsx` o `data/raw/siniestros.xlsx`.

## Salidas

- `data/processed/siniestros_limpio_enriquecido.csv`.
- Registros de ejecución asociados al proceso de limpieza y transformación.


Preprocessing y Feature Engineering

Este notebook implementa la etapa de transformaciones del dataset de víctimas de siniestros viales en CABA. El objetivo es construir una version limpia y enriquecida, reproducible de punta a punta, sin modificar los datos crudos.

## Criterio de arquitectura de datos

Se separa `data/raw/` de `data/processed/` para preservar la fuente original como evidencia inmutable del dato recibido. Esta practica permite auditar decisiones, repetir el pipeline desde cero y comparar resultados si cambian las reglas de limpieza. En esta etapa solo se lee desde `data/raw/` y todo resultado derivado se guarda en `data/processed/`.

In [ ]:
from pathlib import Path
import logging
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import cargar_dataset

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
LOG_DIR = PROJECT_ROOT / "logs"
OUTPUT_FILE = PROCESSED_DATA_DIR / "siniestros_limpio_enriquecido.csv"
LOG_FILE = LOG_DIR / "pipeline.log"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

logger = logging.getLogger("preprocessing_siniestros")
logger.setLevel(logging.INFO)
logger.handlers.clear()

file_handler = logging.FileHandler(LOG_FILE, mode="w", encoding="utf-8-sig")
file_handler.setFormatter(
    logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s")
)
logger.addHandler(file_handler)

stream_handler = logging.StreamHandler()
stream_handler.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
logger.addHandler(stream_handler)

logger.info("Inicio del preprocessing")

## Carga del dataset fuente

La carga se realiza desde el Excel original disponible en `data/raw/`, preferentemente `siniestros_viales_víctimas.xlsx` si existe y, en su defecto, `siniestros.xlsx`. El dataset se copia inmediatamente (`df = df_raw.copy()`) para evitar cualquier modificacion accidental sobre la referencia cargada.

In [ ]:
raw_candidates = [
    RAW_DATA_DIR / "siniestros_viales_victimas.xlsx",
    RAW_DATA_DIR / "siniestros.xlsx",
]
raw_file = next((path for path in raw_candidates if path.exists()), None)

if raw_file is None:
    available_files = sorted(
        path for path in RAW_DATA_DIR.iterdir()
        if path.suffix.lower() in {".xlsx", ".xls", ".csv"}
    )
    if not available_files:
        raise FileNotFoundError("No se encontro un archivo Excel o CSV en data/raw/.")
    raw_file = available_files[0]

try:
    logger.info("Carga de datos desde %s", raw_file)
    df_raw = cargar_dataset(raw_file)
    df = df_raw.copy()
    logger.info("Cantidad inicial: %s filas x %s columnas", df.shape[0], df.shape[1])
except Exception:
    logger.exception("Error durante la carga de datos")
    raise

print(f"Archivo fuente: {raw_file.resolve()}")
print(f"Shape inicial: {df.shape}")
df.head()

## Limpieza de datos


## Limpieza y normalizacion

`SD` y valores equivalentes se interpretan como datos faltantes porque en el diccionario del dataset representan ausencia de informacion registrada, no una categoria sustantiva del fenomeno vial. Mantenerlos como categoria podría sesgar conteos y modelos al hacer que la falta de dato parezca una propiedad real de la victima o del siniestro.

También se convierten tipos relevantes, se normalizan categorías textuales con espacios recortados y mayusculas, se revisan duplicados y se eliminan columnas no utilizadas para modelado. `fecha_fallecimiento_victima` se elimina porque se conoce despues del evento y puede inducir data leakage al anticipar informacion directamente asociada al desenlace mortal.

In [ ]:
try:
    logger.info("Inicio de limpieza y normalizacion")

    original_columns = df.columns.tolist()
    if "GRAVEdad_victima" in df.columns and "gravedad_victima" not in df.columns:
        df = df.rename(columns={"GRAVEdad_victima": "gravedad_victima"})
        logger.info("Columna GRAVEdad_victima renombrada a gravedad_victima")

    missing_equivalents = {
        "", "SD", "S/D", "SIN DATO", "SIN DATOS", "NO DATA", "N/D", "ND", "NA", "NAN", "NONE", "NULL"
    }

    text_columns = df.select_dtypes(include=["object", "string"]).columns.tolist()
    for column in text_columns:
        normalized = df[column].astype("string").str.strip()
        normalized_upper = normalized.str.upper()
        df[column] = normalized_upper.mask(normalized_upper.isin(missing_equivalents), pd.NA)

    if "edad_victima" in df.columns:
        df["edad_victima"] = pd.to_numeric(df["edad_victima"], errors="coerce")

    if "fecha_siniestro" in df.columns:
        df["fecha_siniestro"] = pd.to_datetime(df["fecha_siniestro"], errors="coerce")

    duplicated_rows = int(df.duplicated().sum())
    logger.info("Filas duplicadas detectadas: %s", duplicated_rows)

    columns_to_drop = [
        column for column in ["id_siniestro", "fecha_fallecimiento_victima"]
        if column in df.columns
    ]
    df = df.drop(columns=columns_to_drop)
    logger.info("Columnas eliminadas para modelado: %s", columns_to_drop)
    logger.info("Transformaciones aplicadas: SD a NaN, edad numerica, fecha datetime, texto strip/upper, revision de duplicados")

except Exception:
    logger.exception("Error durante la limpieza y normalizacion")
    raise

print(f"Columnas originales: {original_columns}")
print(f"Duplicados detectados: {duplicated_rows}")
print(f"Columnas eliminadas: {columns_to_drop}")
print(f"Shape tras limpieza: {df.shape}")
df.head()

## Ingeniería de características


## Variables derivadas

Las variables derivadas resumen informacion de alto valor analitico para etapas posteriores. Los grupos etarios reducen ruido y facilitan comparaciones; las variables temporales capturan estacionalidad y patrones semanales; la vulnerabilidad del usuario incorpora conocimiento de dominio sobre exposicion y proteccion relativa en la via publica.

`es_mortal` puede funcionar como target predictivo porque representa un desenlace binario asociado a la severidad del siniestro. Para un segúndo enfoque, `es_grave_o_mortal` permite modelar eventos de alta severidad agrupando casos graves y mortales frente a lesiones leves.

In [ ]:
def asignar_grupo_edad(edad):
    if pd.isna(edad):
        return "sin_dato"
    if edad < 18:
        return "menor_18"
    if edad <= 30:
        return "18_30"
    if edad <= 45:
        return "31_45"
    if edad <= 60:
        return "46_60"
    if edad <= 75:
        return "61_75"
    return "76_mas"


def asignar_vulnerabilidad(modo):
    alta = {"PEATON", "MOTO", "BICICLETA", "MONOPATIN"}
    media = {"AUTO", "TRANSPORTE PUBLICO", "TAXI", "UTILITARIO", "CAMION"}

    if pd.isna(modo):
        return "DESCONOCIDA"
    if modo in alta:
        return "ALTA"
    if modo in media:
        return "MEDIA"
    return "DESCONOCIDA"

try:
    logger.info("Inicio de feature engineering")

    df["edad_grupo"] = df["edad_victima"].apply(asignar_grupo_edad)

    if "gravedad_victima" not in df.columns:
        raise KeyError("No se encontro la columna gravedad_victima para crear targets.")

    df["es_mortal"] = np.where(df["gravedad_victima"].eq("MORTAL"), 1, 0)
    df["es_grave_o_mortal"] = np.where(df["gravedad_victima"].isin(["GRAVE", "MORTAL"]), 1, 0)

    if "modo_desplazamiento_victima" in df.columns:
        df["vulnerabilidad_usuario"] = df["modo_desplazamiento_victima"].apply(asignar_vulnerabilidad)
    else:
        df["vulnerabilidad_usuario"] = "DESCONOCIDA"

    if "fecha_siniestro" in df.columns:
        df["mes_siniestro"] = df["fecha_siniestro"].dt.month.astype("Int64")
        df["dia_semana_siniestro"] = df["fecha_siniestro"].dt.dayofweek.astype("Int64")
        df["trimestre_siniestro"] = df["fecha_siniestro"].dt.quarter.astype("Int64")
    else:
        df["mes_siniestro"] = pd.Series(pd.NA, index=df.index, dtype="Int64")
        df["dia_semana_siniestro"] = pd.Series(pd.NA, index=df.index, dtype="Int64")
        df["trimestre_siniestro"] = pd.Series(pd.NA, index=df.index, dtype="Int64")

    logger.info("Variables creadas: edad_grupo, es_mortal, es_grave_o_mortal, vulnerabilidad_usuario, mes/dia/trimestre")

except Exception:
    logger.exception("Error durante el feature engineering")
    raise

print(f"Shape tras feature engineering: {df.shape}")
df.head()

## Evaluación


## Validacion del resultado

Antes de persistir el archivo final se revisan dimensiones, tipos, nulos y distribuciones de las variables nuevas. Estas salidas dejan evidencia ejecutable de que el dataset enriquecido conserva consistencia estructural y de que los targets quedaron definidos.

In [ ]:
new_columns = [
    "edad_grupo",
    "es_mortal",
    "es_grave_o_mortal",
    "vulnerabilidad_usuario",
    "mes_siniestro",
    "dia_semana_siniestro",
    "trimestre_siniestro",
]

print("Shape final:", df.shape)
print("\nTipos finales:")
display(df.dtypes.rename("dtype").to_frame())

print("\nNulos finales:")
display(df.isna().sum().rename("nulos").to_frame())

for column in new_columns:
    print(f"\nValue counts - {column}:")
    display(df[column].value_counts(dropna=False).rename("cantidad").to_frame())

print("\nDistribucion target es_mortal:")
display(df["es_mortal"].value_counts(normalize=False).rename("cantidad").to_frame())
display(df["es_mortal"].value_counts(normalize=True).rename("proporcion").to_frame())

print("\nDistribucion target es_grave_o_mortal:")
display(df["es_grave_o_mortal"].value_counts(normalize=False).rename("cantidad").to_frame())
display(df["es_grave_o_mortal"].value_counts(normalize=True).rename("proporcion").to_frame())

## Exportación de resultados


## Guardado del dataset procesado

El archivo final se guarda como `data/processed/siniestros_limpio_enriquecido.csv`. La escritura queda registrada en `logs/pipeline.log` junto con las dimensiones iniciales y finales del proceso.

In [ ]:
try:
    df.to_csv(OUTPUT_FILE, index=False)
    logger.info("Cantidad final: %s filas x %s columnas", df.shape[0], df.shape[1])
    logger.info("Ruta de guardado: %s", OUTPUT_FILE)
    logger.info("Fin del preprocessing")
except Exception:
    logger.exception("Error durante el guardado del dataset procesado")
    raise

print(f"Dataset procesado guardado en: {OUTPUT_FILE.resolve()}")
print(f"Log actualizado en: {LOG_FILE.resolve()}")

# Conclusiones

Se consolidó una versión procesada del dataset mediante normalización, tratamiento de valores faltantes y generación de variables derivadas de severidad, edad y temporalidad.

El archivo resultante constituye la entrada principal para análisis exploratorio avanzado, modelado, validación y dashboard, manteniendo trazabilidad respecto del dato original.
